In [1]:
import pandas as pd

df = pd.read_csv("data/spam.csv", encoding="latin-1")
print(df.head())


     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


In [4]:
print(df.columns)
print(df.shape)


Index(['label', 'text'], dtype='object')
(5572, 2)


In [3]:
df = df[['v1', 'v2']]
df.columns = ['label', 'text']


In [5]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})


In [6]:
print(df.head())
print(df['label'].value_counts())


   label                                               text
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...
label
0    4825
1     747
Name: count, dtype: int64


In [7]:
import re
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\royan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [8]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)


In [9]:
df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']].head()


,text,clean_text
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think goes usf lives around though


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

# vectorizer = TfidfVectorizer(max_features=3000)
# X = vectorizer.fit_transform(df['clean_text'])
# y = df['label']



In [11]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.2,
    random_state=42
)
vectorizer = TfidfVectorizer(max_features=3000)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)


In [12]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, y_train)


,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [14]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[965   0]
 [ 29 121]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       965
           1       1.00      0.81      0.89       150

    accuracy                           0.97      1115
   macro avg       0.99      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115



In [15]:
def predict_email(email):
    cleaned = clean_text(email)
    vector = vectorizer.transform([cleaned])
    result = model.predict(vector)
    return "SCAM" if result[0] == 1 else "REAL"


In [16]:
predict_email("Congratulations! You have won free cash, click now!")



'SCAM'

In [ ]:
import pickle

pickle.dump(model, open("model/model.pkl", "wb"))
pickle.dump(vectorizer, open("model/vectorizer.pkl", "wb"))

In [18]:
predict_email("Sir, please find the attached project report.")

'REAL'

In [19]:
predict_email("Your bank account has been temporarily suspended. Verify your details within 24 hours to restore access.")

'REAL'

In [20]:
predict_email("Congratulations! You have been shortlisted for a work-from-home job. No interview required. Earn ₹25,000 per week.")

'REAL'

In [21]:

predict_email("You won an international lottery! Send your details to receive the prize money.")

'SCAM'

In [22]:

predict_email("FINAL NOTICE: Click now to avoid permanent account deactivation.")

'SCAM'

In [23]:
predict_email("Sir, please find the attached lab report for today’s submission.")



'REAL'

In [24]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


Accuracy: 0.9739910313901345


In [25]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)


[[965   0]
 [ 29 121]]


In [26]:
predict_email("""Subject: Assignment Submission Reminder

Hi Ankit,

This is a reminder that the Data Structures assignment must be submitted by
Friday, 5 PM on the LMS portal. Late submissions will be penalized as per
department rules.

If you have already submitted, please ignore this message.

Regards,
Department of Computer Science""")


'REAL'

In [27]:
predict_email("""Subject: Monthly Account Statement Available

Dear Customer,

Your monthly bank account statement for September is now available.
Please log in to the official mobile banking app to view or download it.

For security reasons, we do not share links in emails.

Thank you,
Customer Support Team
""")

'SCAM'

In [28]:
predict_email("""Subject: Interview Schedule – Software Intern Role

Hello Ankit,

Thank you for applying for the Software Intern position.
Your interview has been scheduled for 3rd October at 11:00 AM.

The meeting link will be shared 30 minutes before the interview.

Best regards,
HR Team
""")

'REAL'

In [29]:
predict_email("""Subject: Congratulations! You Have Won ₹5,00,000

Dear User,

You have been selected as the lucky winner of our annual reward program.
To claim your prize, verify your account immediately by clicking the link below.

Act fast! Offer expires today.

Regards,
Rewards Team""")

'SCAM'

In [30]:
predict_email("""Subject: Account Suspended – Immediate Action Required

Your account has been temporarily suspended due to suspicious activity.
Failure to verify within 24 hours will result in permanent closure.

Confirm your details now to avoid service disruption.
""")


'REAL'

In [31]:
predict_email("""Subject: Password Changed Successfully

This is to inform you that your password was successfully changed.
If you did not make this change, please contact customer support immediately.

Thank you,
Security Team""")


'REAL'

In [32]:
predict_email("""Subject: Important Update Regarding Your Account

Dear Valued User,

We noticed unusual login activity on your account and recommend
reviewing your recent activity for safety purposes.

Please sign in to ensure uninterrupted access.

Sincerely,
Support Team
""")

'SCAM'